In [1]:
pip install flwr


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 782.6/782.6 kB 13.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 81.5 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.7/6.7 MB 118.1 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.5/323.5 kB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.7/251.7 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.4/47.4 kB 4.2 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.5
    Uninstalling protobuf-5.29.5:
      Successfully uninstalled protobuf-5.29.5
  Attempting uninstall: pathspec
    Found existing installation: pathspec 1.0.4
    Uninstalling pathspec-1.0.4:
      Successfully uninstalled pathspec-1.0.4
  Attempting uninstall: grpcio
    Found existing installation: grpcio 1.76.0
    Uninstalling grpcio-1.76.0:
      Successfully uninstalled grpcio-1.76.0
  Attempting uninstall: cr

In [2]:
# ==============================
# FLWR (KAGGLE SAFE FIXED)
# ==============================
import flwr as fl
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# -----------------------------
# DATA
# -----------------------------
paths = {
    "B": "/kaggle/input/datasets/mishauroojkhan/dt-driver-wise-data/DT_Driver_Wise_Data/Driver_B_Train.csv",
    "D": "/kaggle/input/datasets/mishauroojkhan/dt-driver-wise-data/DT_Driver_Wise_Data/Driver_D_Train.csv",
    "F": "/kaggle/input/datasets/mishauroojkhan/dt-driver-wise-data/DT_Driver_Wise_Data/Driver_F_Train.csv",
}

CLIENTS = ["B", "D", "F"]

def load_data(path):
    df = pd.read_csv(path)
    X = df.drop(columns=["Class", "Trip_ID"]).values.astype(np.float32)
    y = df["Class"].map({"B": 0, "D": 1, "F": 2}).values
    return X, y

client_data = {c: load_data(paths[c]) for c in CLIENTS}
input_dim = client_data["B"][0].shape[1]

# -----------------------------
# MODEL
# -----------------------------
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, 32)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(32, 3)

    def forward(self, x):
        return self.fc2(self.relu(self.fc1(x)))

# -----------------------------
# METRICS
# -----------------------------
def compute_metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
    }

# -----------------------------
# CLIENT
# -----------------------------
class DriverClient(fl.client.NumPyClient):
    def __init__(self, cid):
        self.model = Net()
        self.X, self.y = client_data[cid]

    def get_parameters(self, config):
        return [v.detach().cpu().numpy() for v in self.model.state_dict().values()]

    def set_parameters(self, parameters):
        params_dict = zip(self.model.state_dict().keys(), parameters)
        state_dict = {k: torch.tensor(v) for k, v in params_dict}
        self.model.load_state_dict(state_dict)

    def fit(self, parameters, config):
        self.set_parameters(parameters)
        self.model.train()

        X = torch.tensor(self.X)
        y = torch.tensor(self.y, dtype=torch.long)

        opt = optim.Adam(self.model.parameters(), lr=1e-3)
        loss_fn = nn.CrossEntropyLoss()

        for _ in range(2):
            opt.zero_grad()
            out = self.model(X)
            loss = loss_fn(out, y)
            loss.backward()
            opt.step()

        return self.get_parameters(config), len(X), {}

    def evaluate(self, parameters, config):
        self.set_parameters(parameters)
        self.model.eval()

        X = torch.tensor(self.X)
        y = self.y

        with torch.no_grad():
            out = self.model(X)
            preds = torch.argmax(out, dim=1).numpy()

        metrics = compute_metrics(y, preds)

        return float(1 - metrics["accuracy"]), len(X), metrics

# -----------------------------
# CLIENT FN
# -----------------------------
def client_fn(cid):
    return DriverClient(CLIENTS[int(cid)])

# -----------------------------
# STRATEGY (IMPORTANT FIX)
# -----------------------------
def aggregate_metrics(metrics):
    # FIX: prevents empty results
    total = sum(n for n, _ in metrics)
    result = {}

    for key in metrics[0][1].keys():
        result[key] = sum(n * m[key] for n, m in metrics) / total

    print("Aggregated:", result)
    return result

strategy = fl.server.strategy.FedAvg(
    fraction_fit=1.0,
    min_fit_clients=3,
    min_available_clients=3,
    evaluate_metrics_aggregation_fn=aggregate_metrics,
)

# -----------------------------
# RUN
# -----------------------------
history = fl.simulation.start_simulation(
    client_fn=client_fn,
    num_clients=3,
    config=fl.server.ServerConfig(num_rounds=5),
    strategy=strategy,
)

# -----------------------------
# SAFE RESULT EXTRACTION (FIXED)
# -----------------------------
acc = history.metrics_distributed.get("accuracy", [])
prec = history.metrics_distributed.get("precision", [])
rec = history.metrics_distributed.get("recall", [])
f1 = history.metrics_distributed.get("f1", [])

if len(acc) > 0:
    results_df = pd.DataFrame({
        "Round": list(range(1, len(acc)+1)),
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1": f1
    })
    print(results_df)

    print("\nFinal:", results_df.iloc[-1])
else:
    print("⚠️ No metrics collected — check aggregation.")

2026-03-18 10:59:58.573799: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773831598.780358      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773831598.841539      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773831599.289450      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773831599.289518      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773831599.289521      55 computation_placer.cc:177] computation placer alr

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

	Instead, use the `flwr run` CLI command to start a local simulation in your Flower app, as shown for example below:

		$ flwr new  # Create a new Flower app from a template

		$ flwr run  # Run the Flower app in Simulation Mode

	Using `start_simulation()` is deprecated.

            This is a deprecated feature. It will be removed
            entirely in future versions of Flower.
        
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
INFO :      Starting Flower simulation, config: num_rounds=5, no round_timeout
2026-03-18 11:00:23,468	INFO worker.py:2007 -- Started a local Ray instance.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is depreca

⚠️ No metrics collected — check aggregation.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
(ClientAppActor pid=246) WARNING :   DEPRECATED FEATURE: `client_fn` now expects a signature `def client_fn(context: Context)`.The provided `client_fn` has signature: {'cid': <Parameter "cid">}. You can import the `Context` like this: `from flwr.common import Context`
(ClientAppActor pid=246) 
(ClientAppActor pid=246)             This is a deprecated feature. It will be removed
(ClientAppActor pid=246)             entirely in future versions of Flower.
(ClientAppActor pid=246)         
(ClientAppActor pid=246) WARNING :   Deprecation Warning: The `client_fn` function must return an instance of `Client`, but an instance of `NumpyClient` was returned. Please use `NumPyClie

In [4]:
# ==============================
# FULL PIPELINE (FINAL VERSION)
# ==============================
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.preprocessing import StandardScaler

# -----------------------------
# PATHS
# -----------------------------
paths = {
    "B": "/kaggle/input/datasets/mishauroojkhan/dt-driver-wise-data/DT_Driver_Wise_Data/Driver_B_Train.csv",
    "D": "/kaggle/input/datasets/mishauroojkhan/dt-driver-wise-data/DT_Driver_Wise_Data/Driver_D_Train.csv",
    "F": "/kaggle/input/datasets/mishauroojkhan/dt-driver-wise-data/DT_Driver_Wise_Data/Driver_F_Train.csv",
}

CLIENTS = ["B", "D", "F"]

# -----------------------------
# LOAD + CLEAN DATA
# -----------------------------
def load_data(path):
    df = pd.read_csv(path)

    df = df.dropna(subset=["Class"])  # FIX NaN
    df = df[df["Class"].isin(["B", "D", "F"])]

    X = df.drop(columns=["Class", "Trip_ID"]).values.astype(np.float32)
    y = df["Class"].map({"B": 0, "D": 1, "F": 2}).values

    return X, y

client_data = {c: load_data(paths[c]) for c in CLIENTS}
input_dim = client_data["B"][0].shape[1]

# -----------------------------
# NORMALIZATION (CRITICAL)
# -----------------------------
scaler = StandardScaler()
X_all = np.vstack([client_data[c][0] for c in CLIENTS])
scaler.fit(X_all)

for c in CLIENTS:
    X, y = client_data[c]
    X = scaler.transform(X)
    client_data[c] = (torch.tensor(X, dtype=torch.float32),
                      torch.tensor(y, dtype=torch.long))

# -----------------------------
# MODEL
# -----------------------------
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Linear(64, 3)
        )

    def forward(self, x):
        return self.net(x)

# -----------------------------
# CLASS WEIGHTS (IMPORTANT)
# -----------------------------
all_labels = torch.cat([client_data[c][1] for c in CLIENTS])
class_counts = torch.bincount(all_labels)
weights = 1.0 / class_counts.float()
weights = weights / weights.sum()

# -----------------------------
# METRICS
# -----------------------------
def compute_metrics(y_true, y_pred):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "Recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "F1": f1_score(y_true, y_pred, average="macro", zero_division=0),
    }

# -----------------------------
# TRAIN FUNCTION
# -----------------------------
def train(model, X, y, epochs=10):
    model.train()

    dataset = torch.utils.data.TensorDataset(X, y)
    loader = torch.utils.data.DataLoader(dataset, batch_size=64, shuffle=True)

    opt = optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.CrossEntropyLoss(weight=weights)

    for _ in range(epochs):
        for xb, yb in loader:
            opt.zero_grad()
            out = model(xb)
            loss = loss_fn(out, yb)
            loss.backward()
            opt.step()

    return model

# -----------------------------
# EVALUATE
# -----------------------------
def evaluate(model):
    model.eval()
    preds_all, y_all = [], []

    with torch.no_grad():
        for c in CLIENTS:
            X, y = client_data[c]
            out = model(X)
            preds = torch.argmax(out, dim=1)

            preds_all.extend(preds.numpy())
            y_all.extend(y.numpy())

    return compute_metrics(y_all, preds_all)

# ==============================
# 1️⃣ CENTRALIZED
# ==============================
X_all = torch.cat([client_data[c][0] for c in CLIENTS])
y_all = torch.cat([client_data[c][1] for c in CLIENTS])

central_model = Net()
central_model = train(central_model, X_all, y_all, epochs=15)

central_metrics = evaluate(central_model)

# ==============================
# 2️⃣ FEDERATED (MANUAL)
# ==============================
def fedavg(weights):
    avg = {}
    for k in weights[0].keys():
        avg[k] = sum(w[k] for w in weights) / len(weights)
    return avg

ROUNDS = [3, 5, 10]
fed_results = []

for r in ROUNDS:
    global_model = Net()

    for _ in range(r):
        local_weights = []

        for c in CLIENTS:
            local_model = Net()
            local_model.load_state_dict(global_model.state_dict())

            X, y = client_data[c]
            local_model = train(local_model, X, y, epochs=5)

            local_weights.append(local_model.state_dict())

        global_model.load_state_dict(fedavg(local_weights))

    metrics = evaluate(global_model)
    metrics["Rounds"] = r
    fed_results.append(metrics)

fed_df = pd.DataFrame(fed_results)

# ==============================
# 3️⃣ COMMUNICATION COST
# ==============================
def model_size_kb(model):
    return sum(p.numel() for p in model.parameters()) * 4 / 1024

model_size = model_size_kb(global_model)

# ==============================
# 4️⃣ COMPARISON TABLE
# ==============================
comparison = pd.DataFrame([
    {"Method": "Centralized", **central_metrics},
    {"Method": "Federated (10 rounds)", **fed_results[-1]}
])

print("\n=== CENTRALIZED vs FEDERATED ===\n", comparison)

# ==============================
# 5️⃣ ABLATION (ROUNDS)
# ==============================
print("\n=== ROUNDS ABLATION ===\n", fed_df)

# ==============================
# 6️⃣ COMMUNICATION COST
# ==============================
print("\n=== COMMUNICATION COST ===")
print(f"Model Size per Round: {model_size:.2f} KB")
print(f"Total Cost (10 rounds): {model_size * 10:.2f} KB")


=== CENTRALIZED vs FEDERATED ===
                   Method  Accuracy  Precision    Recall        F1  Rounds
0            Centralized  0.986127   0.985898  0.986236  0.986044     NaN
1  Federated (10 rounds)  0.710594   0.711424  0.716955  0.706521    10.0

=== ROUNDS ABLATION ===
    Accuracy  Precision    Recall        F1  Rounds
0  0.688003   0.686521  0.694188  0.684168       3
1  0.701190   0.698234  0.707240  0.697690       5
2  0.710594   0.711424  0.716955  0.706521      10

=== COMMUNICATION COST ===
Model Size per Round: 38.01 KB
Total Cost (10 rounds): 380.12 KB


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [5]:
# ==============================
# FINAL FEDERATED PIPELINE (STRONG VERSION)
# ==============================
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.preprocessing import StandardScaler

# -----------------------------
# PATHS
# -----------------------------
paths = {
    "B": "/kaggle/input/datasets/mishauroojkhan/dt-driver-wise-data/DT_Driver_Wise_Data/Driver_B_Train.csv",
    "D": "/kaggle/input/datasets/mishauroojkhan/dt-driver-wise-data/DT_Driver_Wise_Data/Driver_D_Train.csv",
    "F": "/kaggle/input/datasets/mishauroojkhan/dt-driver-wise-data/DT_Driver_Wise_Data/Driver_F_Train.csv",
}

CLIENTS = ["B", "D", "F"]

# -----------------------------
# LOAD DATA
# -----------------------------
def load_data(path):
    df = pd.read_csv(path)
    df = df.dropna(subset=["Class"])
    df = df[df["Class"].isin(["B", "D", "F"])]

    X = df.drop(columns=["Class", "Trip_ID"]).values.astype(np.float32)
    y = df["Class"].map({"B": 0, "D": 1, "F": 2}).values

    return X, y

client_data = {c: load_data(paths[c]) for c in CLIENTS}
input_dim = client_data["B"][0].shape[1]

# -----------------------------
# NORMALIZATION
# -----------------------------
scaler = StandardScaler()
X_all = np.vstack([client_data[c][0] for c in CLIENTS])
scaler.fit(X_all)

for c in CLIENTS:
    X, y = client_data[c]
    X = scaler.transform(X)
    client_data[c] = (torch.tensor(X, dtype=torch.float32),
                      torch.tensor(y, dtype=torch.long))

# -----------------------------
# MODEL
# -----------------------------
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 3)
        )

    def forward(self, x):
        return self.net(x)

# -----------------------------
# CLASS WEIGHTS
# -----------------------------
all_labels = torch.cat([client_data[c][1] for c in CLIENTS])
class_counts = torch.bincount(all_labels)
weights = 1.0 / class_counts.float()
weights = weights / weights.sum()

# -----------------------------
# METRICS
# -----------------------------
def compute_metrics(y_true, y_pred):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "Recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "F1": f1_score(y_true, y_pred, average="macro", zero_division=0),
    }

# -----------------------------
# CENTRAL TRAINING
# -----------------------------
def train(model, X, y, epochs=15):
    model.train()
    dataset = torch.utils.data.TensorDataset(X, y)
    loader = torch.utils.data.DataLoader(dataset, batch_size=64, shuffle=True)

    opt = optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.CrossEntropyLoss(weight=weights)

    for _ in range(epochs):
        for xb, yb in loader:
            opt.zero_grad()
            out = model(xb)
            loss = loss_fn(out, yb)
            loss.backward()
            opt.step()

    return model

# -----------------------------
# EVALUATION
# -----------------------------
def evaluate(model):
    model.eval()
    preds_all, y_all = [], []

    with torch.no_grad():
        for c in CLIENTS:
            X, y = client_data[c]
            out = model(X)
            preds = torch.argmax(out, dim=1)

            preds_all.extend(preds.numpy())
            y_all.extend(y.numpy())

    return compute_metrics(y_all, preds_all)

# ==============================
# CENTRALIZED BASELINE
# ==============================
X_all_t = torch.cat([client_data[c][0] for c in CLIENTS])
y_all_t = torch.cat([client_data[c][1] for c in CLIENTS])

central_model = Net()
central_model = train(central_model, X_all_t, y_all_t)

central_metrics = evaluate(central_model)

# ==============================
# GLOBAL BUFFER (CRITICAL FIX)
# ==============================
buffer_size = int(0.1 * len(X_all_t))
X_global = X_all_t[:buffer_size]
y_global = y_all_t[:buffer_size]

# -----------------------------
# FEDAVG
# -----------------------------
def fedavg(weights):
    avg = {}
    for k in weights[0].keys():
        avg[k] = sum(w[k] for w in weights) / len(weights)
    return avg

# -----------------------------
# FEDPROX TRAINING
# -----------------------------
def train_fedprox(model, global_model, X, y, epochs=5, mu=0.01):
    model.train()

    dataset = torch.utils.data.TensorDataset(
        torch.cat([X, X_global]),
        torch.cat([y, y_global])
    )

    loader = torch.utils.data.DataLoader(dataset, batch_size=64, shuffle=True)

    opt = optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.CrossEntropyLoss(weight=weights)

    global_params = list(global_model.parameters())

    for _ in range(epochs):
        for xb, yb in loader:
            opt.zero_grad()
            out = model(xb)
            loss = loss_fn(out, yb)

            prox = 0.0
            for w, w_t in zip(model.parameters(), global_params):
                prox += ((w - w_t.detach())**2).sum()

            loss += mu * prox
            loss.backward()
            opt.step()

    return model

# ==============================
# FEDERATED TRAINING
# ==============================
ROUNDS = [3, 5, 10]
fed_results = []

for r in ROUNDS:
    global_model = Net()

    for _ in range(r):
        local_weights = []

        for c in CLIENTS:
            local_model = Net()
            local_model.load_state_dict(global_model.state_dict())

            X, y = client_data[c]
            local_model = train_fedprox(local_model, global_model, X, y)

            local_weights.append(local_model.state_dict())

        global_model.load_state_dict(fedavg(local_weights))

    metrics = evaluate(global_model)
    metrics["Rounds"] = r
    fed_results.append(metrics)

fed_df = pd.DataFrame(fed_results)

# ==============================
# COMMUNICATION COST
# ==============================
def model_size_kb(model):
    return sum(p.numel() for p in model.parameters()) * 4 / 1024

model_size = model_size_kb(global_model)

# ==============================
# FINAL TABLES
# ==============================
comparison = pd.DataFrame([
    {"Method": "Centralized", **central_metrics},
    {"Method": "Federated (10 rounds)", **fed_results[-1]}
])

print("\n=== CENTRALIZED vs FEDERATED ===\n", comparison)
print("\n=== ROUNDS ABLATION ===\n", fed_df)

print("\n=== COMMUNICATION COST ===")
print(f"Model Size per Round: {model_size:.2f} KB")
print(f"Total Cost (10 rounds): {model_size * 10:.2f} KB")


=== CENTRALIZED vs FEDERATED ===
                   Method  Accuracy  Precision    Recall        F1  Rounds
0            Centralized  0.985618   0.985566  0.985504  0.985449     NaN
1  Federated (10 rounds)  0.882224   0.882925  0.882692  0.882671    10.0

=== ROUNDS ABLATION ===
    Accuracy  Precision    Recall        F1  Rounds
0  0.809806   0.817368  0.808634  0.811010       3
1  0.839034   0.845004  0.838392  0.839845       5
2  0.882224   0.882925  0.882692  0.882671      10

=== COMMUNICATION COST ===
Model Size per Round: 38.01 KB
Total Cost (10 rounds): 380.12 KB


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
